# Predicting Irrigation Need

In [57]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import classification_report, balanced_accuracy_score

## Load dataset

In [58]:
train = pd.read_csv('./playground-series-s6e4/train.csv')
test = pd.read_csv('./playground-series-s6e4/test.csv')
# train.head()

In [41]:
# test.tail()

In [42]:
# train.info()

In [43]:
# train.describe()

In [44]:
# train.isna().sum()

In [45]:
# for col in train.columns:
# 	if train[col].dtype == 'object':
# 		plt.figure(figsize=(10, 4))
# 		plt.pie(train[col].value_counts(), labels=train[col].value_counts().index, autopct='%1.1f%%')
# 		plt.title(f"Distribution of {col}")
# 		plt.show()

## Cleaning data for EDA:
- Drop column id


In [59]:
test_id = test['id']
for data in [train, test]:
	data.drop(columns=['id'], inplace=True)

In [47]:
# corr = train.corr(numeric_only=True)
# figure, ax = plt.subplots(figsize=(10, 8))
# sns.heatmap(ax=ax, data=corr, annot=True, cmap='coolwarm')

## Cleaning data for modelling

In [60]:
X = train.drop(columns=['Irrigation_Need'])
Y = train['Irrigation_Need']

In [61]:
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=42)

In [62]:
cat_cols = [col for col in X_train.columns if X_train[col].dtype == 'object']
num_cols = [col for col in X_train.columns if X_train[col].dtype != 'object']

In [68]:
preprocessor_tree = ColumnTransformer(
	transformers=[
		('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat_cols),
	],
	remainder='passthrough'
)

preprocessor_linear = ColumnTransformer(
	transformers=[
		('cat', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), cat_cols),
		('num', StandardScaler(), num_cols)
	],
	remainder='passthrough'
)

pipe = Pipeline([
	('preprocessor', preprocessor_linear), # placeholder, will be set in GridSearchCV
	('model', LogisticRegression()) # placeholder, will be set in GridSearchCV
])

In [70]:
param_grid = [

    # 🌲 Random Forest (no scaling)
    {
        'preprocessor': [preprocessor_tree],
        'model': [RandomForestClassifier()],
        'model__n_estimators': [50, 100],
        'model__max_depth': [None, 10]
    },

    # 🌳 Decision Tree (no scaling)
    {
        'preprocessor': [preprocessor_tree],
        'model': [DecisionTreeClassifier()],
        'model__max_depth': [None, 10]
    },

    # 📈 Logistic Regression (needs scaling)
    {
        'preprocessor': [preprocessor_linear],
        'model': [LogisticRegression(max_iter=1000)],
        'model__C': [0.1, 1, 10]
    },

    # # 🧠 SVM (needs scaling)
    # {
    #     'preprocessor': [preprocessor_linear],
    #     'model': [SVC()],
    #     'model__C': [0.1, 1, 10],
    #     'model__kernel': ['rbf', 'linear']
    # }
]

grid = GridSearchCV(
    pipe,
    param_grid,
    cv=3,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train, Y_train)


Fitting 3 folds for each of 9 candidates, totalling 27 fits
[CV] END model=RandomForestClassifier(), model__max_depth=10, model__n_estimators=50, preprocessor=ColumnTransformer(remainder='passthrough',
                  transformers=[('cat',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage',
                                  'Season', 'Irrigation_Type', 'Water_Source',
                                  'Mulching_Used', 'Region'])]); total time= 1.4min
[CV] END model=RandomForestClassifier(), model__max_depth=10, model__n_estimators=50, preprocessor=ColumnTransformer(remainder='passthrough',
                  transformers=[('cat',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['Soil_Type', 'Cr

Fitting 3 folds for each of 9 candidates, totalling 27 fits
[CV] END model=RandomForestClassifier(), model__max_depth=10, model__n_estimators=50, preprocessor=ColumnTransformer(remainder='passthrough',
                  transformers=[('cat',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage',
                                  'Season', 'Irrigation_Type', 'Water_Source',
                                  'Mulching_Used', 'Region'])]); total time= 1.4min
[CV] END model=RandomForestClassifier(), model__max_depth=10, model__n_estimators=50, preprocessor=ColumnTransformer(remainder='passthrough',
                  transformers=[('cat',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['Soil_Type', 'Cr

,estimator,Pipeline(step...egression())])
,param_grid,"[{'model': [RandomForestClassifier()], 'model__max_depth': [None, 10], 'model__n_estimators': [50, 100], 'preprocessor': [ColumnTransfo..., 'Region'])])]}, {'model': [DecisionTreeClassifier()], 'model__max_depth': [None, 10], 'preprocessor': [ColumnTransfo..., 'Region'])])]}, ...]"
,scoring,'f1_macro'
,n_jobs,-1
,refit,True
,cv,3
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('cat', ...)]"


In [71]:
print(f"Best Parameters: {grid.best_params_}")

Best Parameters: {'model': DecisionTreeClassifier(), 'model__max_depth': 10, 'preprocessor': ColumnTransformer(remainder='passthrough',
                  transformers=[('cat',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage',
                                  'Season', 'Irrigation_Type', 'Water_Source',
                                  'Mulching_Used', 'Region'])])}


In [73]:
best_pipeline = grid.best_estimator_
print(f"Best pipeline steps: {best_pipeline.named_steps}")
Y_pred = best_pipeline.predict(X_val)
print(classification_report(Y_val, Y_pred))
print(f"Balanced Accuracy Score: {balanced_accuracy_score(Y_val, Y_pred):.4f}")

Best pipeline steps: {'preprocessor': ColumnTransformer(remainder='passthrough',
                  transformers=[('cat',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage',
                                  'Season', 'Irrigation_Type', 'Water_Source',
                                  'Mulching_Used', 'Region'])]), 'model': DecisionTreeClassifier(max_depth=10)}
              precision    recall  f1-score   support

        High       0.96      0.91      0.93      4249
         Low       0.99      1.00      0.99     73737
      Medium       0.98      0.98      0.98     48014

    accuracy                           0.98    126000
   macro avg       0.98      0.96      0.97    126000
weighted avg       0.98      0.98      0.98    126000

Balanced Accuracy Score: 0.9598


### A quick model

In [63]:
# model = DecisionTreeClassifier(random_state=42)
# model.fit(X_train, Y_train)
# Y_pred = model.predict(X_val)
# print(classification_report(Y_val, Y_pred))
# print(f"Balanced Accuracy Score: {balanced_accuracy_score(Y_val, Y_pred):.4f}")


DT:  
Best Parameters: {'criterion': 'entropy', 'max_depth': 15, 'min_samples_leaf': 4, 'min_samples_split': 10}

In [51]:
model = DecisionTreeClassifier(
    criterion=grid_search.best_params_['criterion'],
    max_depth=grid_search.best_params_['max_depth'],
    min_samples_split=grid_search.best_params_['min_samples_split'],
    min_samples_leaf=grid_search.best_params_['min_samples_leaf'],
    random_state=42)
model.fit(X_train, Y_train)
Y_pred = model.predict(X_val)
print(classification_report(Y_val, Y_pred))
print(f"Balanced Accuracy Score: {balanced_accuracy_score(Y_val, Y_pred):.4f}")

              precision    recall  f1-score   support

           0       0.93      0.91      0.92      4249
           1       0.99      0.99      0.99     73737
           2       0.98      0.97      0.98     48014

    accuracy                           0.98    126000
   macro avg       0.97      0.96      0.96    126000
weighted avg       0.98      0.98      0.98    126000

Balanced Accuracy Score: 0.9589


### Train on all data before prediction:

In [75]:
X_tv = pd.concat([X_train, X_val], axis=0)
Y_tv = np.concatenate((Y_train, Y_val), axis=0)

best_pipeline.fit(X_tv, Y_tv)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [76]:
prediction = best_pipeline.predict(test)
output = pd.DataFrame({'id': test_id, 'Irrigation_Need': prediction})
output.head()

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


In [77]:
output.to_csv('submission4.csv', index=False)